# Small Models, Same Rules - clean Colab run

Runs the full audit grid end to end. **Run the cells top to bottom.**

What it does:
1. Sets up the registered code, dependencies, and gated-model access.
2. Runs the grid **one model at a time** - each model is generated once and scored by all three judges (keyword, Llama-Guard-3-1B, HarmBench classifier) in a single pass.
3. Copies each model's results to Google Drive as it finishes, so a crash only costs the model that was running - not the whole grid.
4. Merges the four models and runs the analysis (tables, confidence intervals, figures).

**Before you start:** use a **freshly rotated** Hugging Face token (the earlier one was exposed), and make sure your pre-registration tag is already public (it is).

## Step 0 - Pick a GPU

Runtime -> Change runtime type -> **L4 GPU** (recommended) or **A100**. The T4 throttled and crashed on the long runs. L4 has 22 GB (no out-of-memory on the 13B judge) and is much faster. The next cell shows which GPU you got.

In [ ]:
!nvidia-smi

## Step 1 - Clone the registered code

Clones your repo at the pre-registered commit `prereg-v1` (the exact frozen code). The `rm -rf` makes this safe to re-run after a reset, and absolute paths mean the working directory is always correct.

In [ ]:
REPO_URL = "https://github.com/shubmittal/Jailbreak-Robustness-Small-LLMs.git"
!rm -rf /content/repo
!git clone $REPO_URL /content/repo
%cd /content/repo
!git checkout prereg-v1
!git log -1 --oneline

## Step 2 - Install dependencies

Takes a couple of minutes.

In [ ]:
!pip install -q -r requirements.txt

## Step 3 - Hugging Face login

First, on huggingface.co, **accept the license** on each gated model page (signed in as the same account your token belongs to):
- `meta-llama/Llama-3.2-3B-Instruct`
- `meta-llama/Llama-Guard-3-1B`
- `google/gemma-2-2b-it`  <- the one missed last time (Google's license is separate from Meta's)

Then run this and paste a **freshly rotated** token. (Do **not** hardcode the token in a cell - the old notebook had it in plaintext and that token is now compromised; rotate it at huggingface.co/settings/tokens.)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Step 4 - Mount Google Drive

Results are copied here after each model, so nothing is lost if the runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/jailbreak_results/run

## Step 5 - Smoke test (~2 min) - canary on Gemma

Runs 2 prompts through **Gemma** specifically, because Gemma is the model whose license failed last time. If this cell loads Gemma and finishes, the license is accepted and the long run will work. Throwaway output.

In [ ]:
!python 05_experiment.py --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword --n 2 --max_new_tokens 16 --no_plot --output_dir ./results/smoke

## Step 6 - Run the grid, one model per cell

Each cell generates 800 completions (200 HarmBench + 200 XSTest, x 2 prompt conditions), scores them with all three judges, writes `results.csv` locally, then copies it to Drive.

Run them in order. **If one crashes, just re-run that one cell** - the others are already safe on Drive. Each is roughly 30-60 min on an L4.

In [ ]:
!python 05_experiment.py --backend transformers --models meta-llama/Llama-3.2-3B-Instruct --model-revisions 0cb88a4f764b7a12671c53f0838cd831a0843b95 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --output_dir ./results/llama
!cp -r ./results/llama /content/drive/MyDrive/jailbreak_results/run/
print('Llama-3.2-3B done + backed up to Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models microsoft/Phi-3-mini-4k-instruct --model-revisions f39ac1d28e925b323eae81227eaba4464caced4e --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --output_dir ./results/phi3
!cp -r ./results/phi3 /content/drive/MyDrive/jailbreak_results/run/
print('Phi-3-mini done + backed up to Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models Qwen/Qwen2.5-3B-Instruct --model-revisions aa8e72537993ba99e69dfaafa59ed015b17504d1 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --output_dir ./results/qwen
!cp -r ./results/qwen /content/drive/MyDrive/jailbreak_results/run/
print('Qwen2.5-3B done + backed up to Drive')

In [ ]:
!python 05_experiment.py --backend transformers --models google/gemma-2-2b-it --model-revisions 299a8560bedf22ed1c72a8a11e7dce4a7f9f51f8 --judges keyword,llamaguard,harmbench --harmbench-cls-size large --n 200 --defense primary --check-prompt-hash 7adca1d95a6759f1eeab9e4ffe45aa5e33ea82a6fe2c57f74c819cd918cf0beb --output_dir ./results/gemma
!cp -r ./results/gemma /content/drive/MyDrive/jailbreak_results/run/
print('Gemma-2-2B done + backed up to Drive')

## Step 7 - Merge the four models into one results.csv

In [ ]:
import pandas as pd, os
root = '/content/drive/MyDrive/jailbreak_results/run'
parts = []
for name in ['llama', 'phi3', 'qwen', 'gemma']:
    p = f'{root}/{name}/results.csv'
    if os.path.exists(p):
        d = pd.read_csv(p); parts.append(d); print(name, len(d), 'rows')
    else:
        print('MISSING:', p)
combined = pd.concat(parts, ignore_index=True)
os.makedirs(f'{root}/combined', exist_ok=True)
combined.to_csv(f'{root}/combined/results.csv', index=False)
print('TOTAL rows:', len(combined), '(expect 9600 = 4 models x 800 x 3 judges)')
print(combined['model'].value_counts())
print(combined['judge'].value_counts())

## Step 8 - Analysis (tables, confidence intervals, figures)

Reads the merged file and writes all analysis outputs into a `results/` subfolder inside the combined Drive folder (the script adds that subfolder itself).

In [ ]:
!python 06_analysis.py --results-csv /content/drive/MyDrive/jailbreak_results/run/combined/results.csv --out-dir /content/drive/MyDrive/jailbreak_results/run/combined --bootstrap 1000 --ci 0.95

## Step 9 - Verify outputs

In [ ]:
!ls -la /content/drive/MyDrive/jailbreak_results/run/combined/results
import json
try:
    s = json.load(open('/content/drive/MyDrive/jailbreak_results/run/combined/results/summary.json'))
    print(json.dumps(s, indent=2)[:2000])
except Exception as e:
    print('summary.json not found yet:', e)

## Troubleshooting & notes

- **A model cell crashes:** re-run just that cell. The finished models are already on Drive.
- **Out-of-memory on the 13B judge:** add `--harmbench-cls-size small` (Mistral-7B) to that model's command.
- **Runtime fully reset:** re-run Steps 1-4, then re-run only the model cells you hadn't finished (completed ones are on Drive), then Steps 7-8.
- **Registration check:** open any `results/<model>/run_manifest.json` and confirm its `timestamp` is later than your GitHub push / arXiv time.
- **Benchmark sizes:** this uses `--n 200`, so XSTest is sampled to 200. Your pre-registration lists XSTest-250 (and OR-Bench-Hard). For the full XSTest-250 use `--n 250`; to add OR-Bench-Hard add `--enable-orbench` (much longer run). Confirm against your registered protocol before deciding.
- **HF token:** rotate it - the previous one was exposed.